# Brain Tumor MRI Classification
### BMEN 207 Honors Project — Dhir Parekh & Bhargav Ashwin

**Model:** ResNet-50 Transfer Learning (Two-Phase)  
**Classes:** Glioma · Meningioma · No Tumor · Pituitary  
**Dataset:** Kaggle Brain MRI Images (~7,000 images)

---
Run cells **in order, top to bottom**. Each section corresponds to a project task.

## ⚙️ Step 0 — Runtime Setup
**Before running anything:** Go to `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Verify GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install -q kaggle opencv-python-headless albumentations grad-cam openpyxl
print('All packages installed.')

## 📁 Step 2 — Clone Repo & Set Working Directory

In [ ]:
import os

REPO_URL  = 'https://github.com/dhir08/BMEN-207-Honors-Proj.git'
REPO_NAME = 'BMEN-207-Honors-Proj'
REPO_PATH = f'/content/{REPO_NAME}'

if not os.path.isdir(REPO_PATH):
    !git clone {REPO_URL}
else:
    print('Repo already cloned — pulling latest changes')
    !cd {REPO_PATH} && git pull

# Set working directory — use %cd so it persists across all cells
%cd {REPO_PATH}
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

## 🔑 Step 3 — Kaggle Credentials
Upload your `kaggle.json` API token.
Get it at: [kaggle.com/settings → API → Create New Token](https://www.kaggle.com/settings)

In [ ]:
from google.colab import files
import os

# Upload kaggle.json when the file picker appears
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle credentials installed ✓')

## 📥 Task 3 — Download Dataset

In [ ]:
!python scripts/download_dataset.py

## 🔍 Task 4 — Audit Dataset

In [ ]:
!python scripts/audit_dataset.py

## 🖼️ Task 5 — Preprocess Images
Resize to 224×224, convert to RGB, normalize.

In [ ]:
!python scripts/preprocess.py

## 🔄 Task 6 — Data Augmentation
Flips, rotations, brightness/contrast jitter via Albumentations.

In [ ]:
!python scripts/augment.py

## ✂️ Task 7 & 8 — Split Dataset into Train / Test Batches
1,400 images/class for training · 4 batches of 100 images/class for testing.

In [ ]:
!python scripts/split_dataset.py

In [ ]:
!python scripts/verify_batches.py

## 🔁 Task 9 — Verify DataLoader

In [ ]:
import sys
sys.path.insert(0, '.')

from scripts.dataloader import get_train_loader, get_test_batch_loader

train_loader = get_train_loader(batch_size=32, num_workers=2)
val_loader   = get_test_batch_loader(batch_num=1, batch_size=32, num_workers=2)

imgs, lbls = next(iter(train_loader))
print(f'Train — batch shape: {tuple(imgs.shape)}, labels: {lbls[:8].tolist()}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 🏗️ Tasks 10 & 11 — Model Architecture

In [ ]:
!python scripts/model_selection.py

In [ ]:
from scripts.model import build_model, freeze_backbone, describe_model
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = build_model(pretrained=True).to(device)
freeze_backbone(model)
describe_model(model)

## ⚙️ Task 12 — Training Configuration

In [ ]:
!python scripts/train_config.py

## 🚀 Task 13 — Train the Model
**This is the main training cell. Runs Phase 1 (5 epochs) then Phase 2 (up to 20 epochs).**  
Checkpoints are saved after every epoch — safe to resume if the Colab session drops.

In [ ]:
!python scripts/train.py

In [ ]:
# To RESUME from a checkpoint after a session timeout:
# !python scripts/train.py --resume models/checkpoints/<latest_checkpoint>.pt --phase 2

## 📊 Task 14 — Evaluate on All 4 Test Batches

In [ ]:
!python scripts/evaluate.py

## 📈 Task 15 — Confusion Matrix, Precision, Recall, F1

In [ ]:
!python scripts/metrics.py

In [ ]:
# Display confusion matrix inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
img = mpimg.imread('outputs/plots/confusion_matrix.png')
plt.figure(figsize=(7,6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 🔥 Task 16 — Grad-CAM Explainability Heatmaps

In [ ]:
!pip install -q grad-cam
!python scripts/gradcam.py

## 🖼️ Task 18 — Visualize Predictions + Grad-CAM + Training Curves

In [ ]:
!python scripts/visualize.py

In [ ]:
# Display all-class Grad-CAM summary inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

plots = [
    ('outputs/plots/training_curves.png',       'Training Curves'),
    ('outputs/plots/all_classes_summary.png',   'Grad-CAM Summary (All Classes)'),
]

for path, title in plots:
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.title(title, fontsize=13)
        plt.axis('off')
        plt.tight_layout()
        plt.show()

## 🏆 Task 19 — Benchmark vs Literature Baselines

In [ ]:
!python scripts/benchmark.py

In [ ]:
# Display benchmark bar chart inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
img = mpimg.imread('outputs/plots/benchmark_bar.png')
plt.figure(figsize=(11, 5))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 💾 Save Outputs to Google Drive (Optional)
Prevents losing results when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

DRIVE_DEST = '/content/drive/MyDrive/BMEN207_outputs'
os.makedirs(DRIVE_DEST, exist_ok=True)

# Copy outputs and best model
shutil.copytree('outputs', f'{DRIVE_DEST}/outputs', dirs_exist_ok=True)
if os.path.exists('models/best_model.pt'):
    os.makedirs(f'{DRIVE_DEST}/models', exist_ok=True)
    shutil.copy('models/best_model.pt', f'{DRIVE_DEST}/models/best_model.pt')

print(f'Saved to {DRIVE_DEST}')

## 📋 Full Results Summary

In [ ]:
import json, os

files_to_show = [
    ('outputs/metrics/evaluation_results.json', 'Evaluation Results'),
    ('outputs/metrics/classification_report.json', 'Classification Report'),
    ('outputs/metrics/benchmark_comparison.json', 'Benchmark Comparison'),
]

for path, label in files_to_show:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f'\n=== {label} ===')
        if 'overall' in data:
            print(f'  Overall accuracy: {data["overall"].get("accuracy", "N/A")}')
        if 'per_class' in data:
            for cls, m in data['per_class'].items():
                print(f'  {cls:<15s}: F1={m.get("f1_score","N/A")}  P={m.get("precision","N/A")}  R={m.get("recall","N/A")}')
        if 'our_model' in data:
            our = data['our_model']
            print(f'  Our accuracy: {our.get("accuracy_pct", "Pending")}%')
    else:
        print(f'{label}: not yet generated')